# Week 3 participant practical: Čech and Rips filtrations

This is the student investigation, built around three equal-radius balls followed by one sampled circle. The setup and mathematical framing are supplied. You must record predictions, complete short computational steps, check intermediate objects and justify an interpretation.

This practical keeps the points fixed and changes one construction choice at a time:

$$S\subset\mathbb R^2 \longrightarrow \{B(x,r)\}_{x\in S}
\longrightarrow \check C_r(S)\ \text{or}\ \operatorname{Rips}_r(S)
\longrightarrow H_p(-;\mathbb F_2).$$

We use the **ball-radius convention**: Čech uses balls of radius $r$, while Rips contains simplices of diameter at most $2r$. With this convention the two complexes have the same edges and
$\check C_r(S)\subseteq\operatorname{Rips}_r(S)$.

**† Qualification.** Many libraries instead call the pairwise-distance threshold itself the Rips parameter. Always check whether a reported value is a radius $r$ or a diameter threshold $2r$.

**Working rule.** Run one section at a time. Before each TODO, state what shape, dimension or direction you expect in the output. Optional extensions come only after the core checkpoints agree.



In [ ]:
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

def faces(simplex):
    return [tuple(f) for k in range(1, len(simplex)) for f in combinations(simplex, k)]

def close_under_faces(maximal):
    K = set()
    for s in maximal:
        s = tuple(sorted(s)); K.add(s); K.update(faces(s))
    return K

def boundary_matrix(K, p):
    cols = sorted(s for s in K if len(s) == p + 1)
    rows = sorted(s for s in K if len(s) == p) if p else []
    A = np.zeros((len(rows), len(cols)), dtype=np.uint8)
    lookup = {s:i for i,s in enumerate(rows)}
    if p:
        for j,s in enumerate(cols):
            for f in combinations(s, p): A[lookup[f],j] = 1
    return A

def rank_mod2(A):
    A=np.array(A,dtype=np.uint8,copy=True)%2; row=rank=0
    for col in range(A.shape[1]):
        piv=np.flatnonzero(A[row:,col])
        if not len(piv): continue
        q=row+piv[0]; A[[row,q]]=A[[q,row]]
        for i in range(A.shape[0]):
            if i!=row and A[i,col]: A[i]^=A[row]
        row+=1; rank+=1
        if row==A.shape[0]: break
    return rank

def betti(K, max_dim=1):
    out=[]
    for p in range(max_dim+1):
        n=sum(len(s)==p+1 for s in K)
        rp=rank_mod2(boundary_matrix(K,p)) if p else 0
        rn=rank_mod2(boundary_matrix(K,p+1))
        out.append(n-rp-rn)
    return tuple(out)

def enclosing_radius(points):
    """Radius of the smallest enclosing ball for one, two or three planar points."""
    P=np.asarray(points,float)
    if len(P)==1: return 0.0
    d=np.array([np.linalg.norm(P[i]-P[j]) for i,j in combinations(range(len(P)),2)])
    if len(P)==2: return d[0]/2
    a,b,c=sorted(d)
    if c*c >= a*a+b*b-1e-12: return c/2
    area=abs(np.cross(P[1]-P[0],P[2]-P[0]))/2
    return a*b*c/(4*area)

def rips(points, r, max_dim=2):
    """Rips(r): simplices of diameter at most 2r."""
    K=set()
    for k in range(1,max_dim+2):
        for s in combinations(range(len(points)),k):
            diameter=max([0.0]+[np.linalg.norm(points[i]-points[j]) for i,j in combinations(s,2)])
            if diameter <= 2*r+1e-12: K.add(s)
    return K

def cech(points, r, max_dim=2):
    """Cech(r): balls of radius r centred at the vertices have common intersection."""
    K=set()
    for k in range(1,max_dim+2):
        for s in combinations(range(len(points)),k):
            if enclosing_radius(points[list(s)]) <= r+1e-12: K.add(s)
    return K

def graph_complex(points, r):
    """Only vertices and Rips edges, with no clique filling."""
    return {s for s in rips(points,r,max_dim=1)}

def valid_filtration_value(values):
    violations=[]
    for s,v in values.items():
        for f in faces(s):
            if f in values and values[f] > v: violations.append((f,s))
    return violations

def draw(K, points, ax, title):
    for tri in sorted(s for s in K if len(s)==3):
        ax.fill(*zip(*points[list(tri)]),color='tab:blue',alpha=.22)
    for e in sorted(s for s in K if len(s)==2):
        ax.plot(*zip(*points[list(e)]),color='black',lw=1.5)
    ax.scatter(points[:,0],points[:,1],s=55,zorder=3)
    for i,p in enumerate(points): ax.text(p[0],p[1]+.08,str(i),ha='center')
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')

print('Week 3 helpers ready. Homology is computed over F_2.')
print('Complexes are built through dimension 2, so reported Betti numbers are beta_0 and beta_1.')

## 1. Observe: participant checkpoint

The three points below form an equilateral triangle of side length $1.9$. We examine radius $r=1$. Pairwise balls intersect because every pair of centres is at distance below $2r$. Before constructing a complex, ask the genuinely joint question: do all three balls share one point?

In [ ]:
triangle=np.array([[0.,0.],[1.9,0.],[0.95,1.9*np.sqrt(3)/2]])
r=1.0
theta=np.linspace(0,2*np.pi,300)
fig,ax=plt.subplots(figsize=(5,5))
for i,p in enumerate(triangle):
    ax.plot(p[0]+r*np.cos(theta),p[1]+r*np.sin(theta),alpha=.75)
    ax.scatter(*p); ax.text(p[0],p[1]+.08,str(i),ha='center')
ax.set_aspect('equal'); ax.set_title('Three radius-1 balls'); ax.axis('off'); plt.show()
print('pairwise distances:',np.round([np.linalg.norm(triangle[i]-triangle[j]) for i,j in combinations(range(3),2)],3))
print('smallest common enclosing radius:',round(enclosing_radius(triangle),3))

## 2. Predict: participant checkpoint

Before running the construction:

1. Which vertices and edges should appear in both complexes at $r=1$?
2. Should the 2-simplex $(0,1,2)$ appear in Čech, Rips, both, or neither?
3. Predict $\beta_1$ for the graph alone, the Čech complex, and the full Rips complex.
4. If $r$ increases, can an existing simplex disappear from either filtration? Explain using the definition.

The code constructs simplices only through dimension 2. This is sufficient for $H_0$ and $H_1$, which are the quantities reported here; it is not sufficient for a valid $H_2$ calculation because omitted 3-simplices could fill 2-cycles.

## 3. Implement: participant checkpoint

Čech records common intersections of balls. Rips checks only pairwise distances and fills every clique. The following calculation exposes the difference without calling a TDA library.

In [ ]:
C=cech(triangle,r); R=rips(triangle,r); G=graph_complex(triangle,r)
print('Cech simplices:',sorted(C,key=lambda s:(len(s),s)))
print('Rips simplices:',sorted(R,key=lambda s:(len(s),s)))
# TODO: print betti(G), betti(C), and betti(R)


### Check the filtration property

A filtration value must be monotone on faces. Diagnose the deliberately invalid assignment below. Then repair it by assigning each simplex the maximum value of its faces.

In [ ]:
bad={(0,):0.,(1,):0.,(2,):0.,(0,1):1.,(0,2):1.,(1,2):1.4,(0,1,2):1.2}
print('violations:',valid_filtration_value(bad))
# TODO: repair the triangle value


### Follow one point cloud through scale

Use eight equally spaced points on a circle. Predict the scale at which the components join, the scale at which a loop is present, and whether clique filling eventually kills it.

In [ ]:
angles=np.linspace(0,2*np.pi,8,endpoint=False)
circle=np.c_[np.cos(angles),np.sin(angles)]
radii=[0.20,0.39,0.72,1.01]
# TODO: for each radius, compute and print the Rips Betti numbers
# TODO: draw the four complexes with draw(...)


## 4. Compare: participant checkpoint

At $r=1$, keep the same triangle and the same pairwise edges. Compare three mathematical objects:

1. the graph as a one-dimensional simplicial complex;
2. the Čech complex, requiring joint ball intersection;
3. the Rips clique complex, filling every clique.

Which output difference is caused by scale, and which by the rule for adding higher-dimensional simplices?

In [ ]:
# TODO: draw G, C and R side by side and report their Betti numbers


## 5. Interpret: participant checkpoint

1. Why can the Rips complex fill a triangular loop that remains unfilled in Čech at the same radius?
2. What is preserved by the Nerve Theorem when the cover consists of Euclidean balls?
3. Why is a Rips complex more than a proximity graph?
4. In an interaction network, what scientific assumption is made by clique filling?
5. For a dynamical-system point cloud, what could make Euclidean distance a poor filtration parameter?

**◇ Object check.** The observed object is a finite point cloud. The union of balls, its Čech nerve, the Rips clique complex, and their homology groups are four different constructed objects. A loop belongs to the selected construction, not automatically to the underlying dynamical system.